# 开源仓介绍及验证
CANN开源算子仓对CANN内置算子库进行了全面开源，为开发者提供了丰富的算子参考实现与优化方案。本节将围绕以下内容，介绍CANN开源算子仓的整体结构与验证流程。
- 环境准备
- 仓库说明
- 工程介绍
- 算子验证

---

## **1. 环境准备**
本文所有内容均存放于Sources文件夹。  
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。

In [ ]:
!mkdir -p Sources

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

ops-math仓编译算子工程时依赖gawk，执行以下命令安装gawk

In [ ]:
%%bash
set -e

GAWK_VERSION="5.3.0"
INSTALL_DIR="$HOME/.local"
RC_FILE="$HOME/.bashrc"  # 固定使用.bashrc，不存在则创建
PATH_LINE="export PATH=$INSTALL_DIR/bin:\$PATH"
# 确保.bashrc存在，且只添加一次PATH
touch "$RC_FILE" 
source "$RC_FILE"
if command -v gawk &> /dev/null; then
    echo "✅ gawk 已存在，版本：$(gawk --version | head -n1)"
    echo "✅ 跳过编译安装"
    exit 0
fi

[ ! -f "gawk-$GAWK_VERSION.tar.gz" ] && wget https://ftp.gnu.org/gnu/gawk/gawk-$GAWK_VERSION.tar.gz

tar -zxf gawk-$GAWK_VERSION.tar.gz --overwrite 
cd gawk-$GAWK_VERSION
./configure --prefix=$INSTALL_DIR
make -j$(nproc)
make install

grep -q "$INSTALL_DIR/bin" "$RC_FILE" || echo "$PATH_LINE" >> "$RC_FILE"
source "$RC_FILE"

echo "gawk 安装完成，路径：$(which gawk)"

In [ ]:
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:{os.environ['PATH']}"

## **2. 仓库说明**
CANN内置了丰富的算子库，可与第三方框架算子完成映射，支撑AI框架在昇腾平台上高效运行，也支持用户直接调用以实现计算加速。CANN开源算子仓将这部分核心算子库全面开源，开发者可直接查看内置算子的完整实现逻辑，用于学习参考、二次开发或持续优化。

### 2.1 仓库列表
CANN算子库的整体架构如下图所示：

<img src="./images/architecture.png" alt="pytorch_net"  width="700px" >

CANN开源算子仓将算子库全面开源，提供四类核心仓库。这些仓库包含经过深度优化、具备硬件亲和性的高性能算子，为神经网络在昇腾硬件上的计算加速提供核心基础能力。各仓库的详细信息如下表所示。

<table style="float: left; border-collapse: collapse; margin: 0 10px 10px 0; font-size: 14px;">
<tr style="background-color:#f0f0f0">
  <td align="center"><strong>仓库</strong></td>
  <td align="center"><strong>地址</strong></td>
  <td align="center"><strong>说明</strong></td>
</tr>
<tr>
  <td align="left">	ops-nn </td>
  <td align="left">	https://gitcode.com/cann/ops-nn </td>
  <td align="left"> ops-nn是CANN算子库中提供神经网络计算能力的高阶算子库，包括matmul类、activation类等算子 </td>
</tr>
<tr>
  <td align="left">	ops-math </td>
  <td align="left">	https://gitcode.com/cann/ops-math </td>
  <td align="left"> ops-math是CANN算子库中提供数学类计算的基础算子库，包括math类、conversion类等算子 </td>
</tr>
<tr>
  <td align="left">	ops-transformer </td>
  <td align="left">	https://gitcode.com/cann/ops-transformer </td>
  <td align="left"> ops-transformer是CANN算子库中提供transformer类大模型计算的进阶算子库，包括attention类、moe类等算子 </td>
</tr>
<tr>
  <td align="left">	ops-cv </td>
  <td align="left">	https://gitcode.com/cann/ops-cv </td>
  <td align="left"> ops-cv是CANN算子库中提供图像处理、目标检测等能力的高阶算子库，涵盖常见的图像处理操作，包括image类、objdetect类算子 </td>
</tr>
</table>
<div style="clear: both;"></div>

### 2.2 仓库结构
接下来以 ops-math 仓为例介绍开源算子仓的目录组织方式，其核心结构如下：
```
ops-math
├── cmake                          # 项目工程编译目录
├── common                         # 项目公共头文件和公共源码
├── docs                           # 项目文档介绍
├── examples                       # 端到端算子开发和调用示例
├── experimental                   # 用户自定义算子存放目录
├── math                           # math类算子
│   ├── is_finite                  # is_finite算子所有交付件，如Tiling、Kernel等
│   │   ├── CMakeLists.txt         # 算子编译配置文件
│   │   ├── docs                   # 算子说明文档
│   │   ├── examples               # 算子使用示例
│   │   ├── op_api                 # 可选，算子aclnn接口实现目录，如未提供则表示此算子的aclnn接口会让工程自动生成
│   │   ├── op_graph               # 算子构图相关目录
│   │   ├── op_host                # 算子信息库、Tiling、InferShape相关实现目录
│   │   ├── op_kernel              # 算子kernel目录
│   │   ├── test                   # 算子测试代码
│   │   └── README.md              # 算子介绍文档
│   ├── ...
│   └── CMakeLists.txt             # 算子编译配置文件
├── ...
├── conversion                     # conversion类算子
├── random                         # random类算子
├── scripts                        # 脚本目录，包含自定义算子、kernel构建相关配置文件
├── tests                          # 测试工程目录
├── CMakeLists.txt
├── README.md
├── build.sh                       # 项目工程编译脚本
├── install_deps.sh                # 安装依赖包脚本
└── requirements.txt               # 项目需要的第三方依赖包
```

可执行以下命令下载 ops-math 仓并查看其实际目录结构：

In [ ]:
!git clone https://gitcode.com/cann/ops-math.git
%cd ops-math 
!ls -l ./

当前仓库的核心目录结构及用途如下：
- examples：算子仓单个算子的工程示例目录，用户贡献算子时，需参照该目录下的交付件规范完成交付。
- math：内置数学计算类算子的专属目录，所有原生数学计算类型算子均存放于此。
- random：内置随机类算子的专属目录，所有原生随机类型算子均存放于此。
- conversion：内置类型/格式/布局转换类算子的专属目录，所有原生数据类型转换、张量格式转换、维度布局转换等基础转换类算子均存放于此。
- experimental：用户贡献算子的统一存放目录，该目录下按算子类型拆分出 conversion、math、random 三个子文件夹，分别存放用户贡献的对应类型算子。

可通过修改进入的目录，查看各核心目录下的具体文件内容。

---

## **3. 工程介绍**
由于开源算子仓是内置算子的集合，功能上需要支持多个算子同时编译出包，所以每个算子的交付件和前面介绍过的单算子工程的交付件存在部分差异。

### 3.1 算子工程目录结构
以下介绍仓内的算子的格式，以Ascend C算子为例说明各个核心交付件。
```
├── ${op_class}                                         # 算子分类，如conversion、math、 random类算子
│   ├── ${op_name}                                      # 算子工程目录，${op_name}表示算子名（小写下划线形式）
│   │   ├── CMakeLists.txt                              # 算子cmakelist入口
│   │   ├── README.md                                   # 算子介绍文档
│   │   ├── docs                                        # 可选，算子文档目录
│   │   │   └── aclnn${OpName}.md                       # 可选，算子aclnn接口介绍文档，${OpName}表示算子名（大驼峰形式）
│   │   ├── examples                                    # 算子调用示例目录
│   │   │   ├── test_aclnn_${op_name}.cpp               # 算子通过aclnn调用的示例
│   │   │   └── test_geir_${op_name}.cpp                # 算子通过geir调用的示例
│   │   ├── op_api                                      # 可选，算子aclnn实现文件目录，若未配置工程自动生成
│   │   │   └── ... 
│   │   ├── op_graph                                    # 可选，图融合相关实现
│   │   │   └── ...
│   │   ├── op_host                                     # Host侧实现
│   │   │   ├── CMakeLists.txt                          # Host侧cmakelist文件
│   │   │   ├── config                                  # 可选，二进制配置文件，若未配置工程自动生成
│   │   │   │   └── ...
│   │   │   ├── ${op_name}_def.cpp                      # 算子信息库，定义算子基本信息，如名称、输入输出、数据类型等
│   │   │   ├── ${op_name}_infershape.cpp               # 可选，InferShape实现
│   │   │   ├── ${op_name}_tiling_${sub_case}.cpp       # 可选，针对某些子场景下的Tiling优化
│   │   │   ├── ${op_name}_tiling_${sub_case}.h         # 可选，${sub_case}子场景下Tiling实现用的头文件
│   │   │   ├── ${op_name}_tiling.cpp                   # 可选，若无该文件表明对应场景下无Tiling实现
│   │   │   └──${op_name}_tiling.h                     # 可选，Tiling实现用的头文件
│   │   │── op_kernel                                   # AI Core算子Device侧Kernel实现
│   │   │   ├── ${sub_case}                             # 可选，${sub_case}子场景使用的目录
│   │   │   ├── ${op_name}_tiling_key.h                 # 可选，TilingKey文件，定义Tiling策略的Key
│   │   │   ├── ${op_name}_tiling_data.h                # 可选，TilingData文件，存储Tiling策略相关配置信息
│   │   │   ├── ${op_name}.cpp                          # Kernel入口文件，包含主函数和调度逻辑
│   │   │   └── ${op_name}.h                            # Kernel实现文件，定义Kernel头文件，包含函数声明、结构定义、逻辑实现
│   │   └── tests                                       # 算子测试用例目录
│   │       ├── CMakeLists.txt
│   │       └── ut                                      # 可选，UT测试用例，根据实际情况开发相应的用例
│   │           ├── CMakeLists.txt                      # UT用例cmakelist文件
│   │           ├── op_host                             # op_host测试用例目录
│   │           │   └── ...  
│   │           └── op_kernel                           # op_kernel测试用例目录
│   │               └── ...  
│   └── ...
```

可以看到，核心交付件还是op_host和op_kernel目录下的文件。

### 3.2 examples中add算子交付件展示
接下来以examples中的add算子为例介绍对应交付件。

执行以下命令可以看到examples中的标准交付件。

In [ ]:
!cd ./examples/add_example;find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

- **examples/test_aclnn_add_example.cpp**：交付必要文件，进行算子API调用测试。
- examples/test_geir_add_example.cpp：交付可选文件，算子入图调用测试。
- op_graph/add_example_graph_infer.cpp：交付可选文件，进行算子数据类型推导。
- op_graph/add_example_proto.h：交付可选文件，算子原型定义，用于图优化和融合阶段识别算子。
- op_graph/fusion_pass：交付可选文件，融合规则文件夹。
- **op_host/add_example_def.cpp**：交付必要文件，算子信息库。
- **op_host/add_example_infershape.cpp**：交付必要文件，算子形状推导，确定输出张量形状。
- **op_host/add_example_tiling.cpp**：交付必要文件，算子切分逻辑，适配硬件计算资源。
- op_host/config：交付可选目录，编译期间会自动生成。
- **op_kernel/add_example.cpp**：交付必要文件，算子核心计算逻辑实现。
- **op_kernel/add_example.h**：交付必要文件，算子核心头文件，声明核心函数与数据结构。
- **op_kernel/add_example_tiling_data.h**：交付必要文件，tiling结构体定义文件。
- **op_kernel/add_example_tiling_key.h**：交付必要文件，tilingkey定义文件。
- CMakeLists.txt：交付必要文件，编译自动生成。
- **README.md**：交付可选文件，算子使用、编译、测试说明文档，需要按照格式编写。
- tests/ut/op_host：交付可选文件，op_host侧代码单元测试。
- tests/ut/op_kernel：交付可选文件，op_kernel侧代码单元测试。

可执行以下代码查看README.md文件具体内容，并自行修改查看add_example中各个文件的具体代码。

In [ ]:
!cat ./examples/add_example/README.md

---

## **4. 算子验证**
了解仓库结构与算子工程结构后，接下来就是使用仓库进行算子的验证。验证方式有以下三种：

- **自定义算子包**：选择部分算子编译生成的包称为自定义算子包，以挂载形式作用于CANN包，不改变原始包内容。生成的自定义算子包优先级高于原始CANN包。该包支持aclnn和图模式调用AI Core、AI CPU算子。

- **ops-math包**：选择整个项目编译生成的包称为ops-math包，可完整替换CANN包对应部分。该包支持aclnn和图模式调用AI Core算子。

- **ops-math静态库**：指整个项目编译为一个静态库文件，包含libcann_math_static.a和aclnn接口头文件。该包仅支持aclnn调用AI Core算子。

本教程以自定义算子包方式进行experimental/math/arange算子的整体验证。

### 4.1 编译自定义算子包
进入项目根目录，执行如下编译命令：

```
bash build.sh --pkg --soc=${soc_version} [--vendor_name=${vendor_name}] [--ops=${op_list}] [-j${n}]
# 以Abs算子编译为例
# bash build.sh --pkg --soc=ascend910b --ops=abs -j16
# 编译experimental贡献目录下的用户算子（以Abs算子为例，编译时请以实际贡献算子为准）
# bash build.sh --pkg --experimental --soc=ascend910b --ops=abs -j16
```

其中：
- --soc：`${soc_version}`表示NPU型号。Atlas A2 训练系列产品/Atlas A2 推理系列产品使用"ascend910b"（默认），Atlas A3 训练系列产品/Atlas A3 推理系列产品使用"ascend910_93"，Ascend 950PR/Ascend 950DT产品使用"ascend950"。
- --vendor_name（可选）：`${vendor_name}`表示构建的自定义算子包名，默认名为custom。
- --ops（可选）：`${op_list}`表示待编译算子，不指定时默认编译所有算子。格式形如"abs,add_lora,..."，多算子之间用英文逗号","分隔。
- --experimental（可选）：表示编译用户保存在experimental贡献目录下的算子。

**注意：--ops和--vendor_name参数需至少传入一个，否则编译的是ops-math包，会替换掉CANN包中的内置算子，并导致本教程后续执行存在问题。**

执行以下代码，进行experimental/math/arange算子编译。

In [ ]:
!bash build.sh --pkg --experimental --vendor_name=custom --soc=ascend910b --ops=arange -j16

当提示如下信息，说明编译成功。

```
Self-extractable archive "cann-ops-math-${vendor_name}_linux-${arch}.run" successfully created.
```

编译成功后，run包存放于项目根目录的build_out目录下。

### 4.2 安装自定义算子包
安装自定义算子包的命令如下所示。

```
./build_out/cann-ops-math-${vendor_name}_linux-${arch}.run
```

自定义算子包安装路径为`${ASCEND_HOME_PATH}`/opp/vendors，`${ASCEND_HOME_PATH}`已通过环境变量配置，表示CANN toolkit包安装路径，一般为`${install_path}`/cann。

执行以下命令进行CANN包安装

In [ ]:
!./build_out/cann-ops-math-custom_linux-aarch64.run --install-path=${HOME}/

### 4.3 基于自定义算子包执行算子样例
包安装后，执行如下命令：

```
bash build.sh --run_example ${op} ${mode} ${pkg_mode} [--vendor_name=${vendor_name}] [--soc=${soc_version}] [--experimental]
# 以Abs算子example执行为例
# bash build.sh --run_example abs eager cust --vendor_name=custom
# 以Abs算子experimental执行为例
# bash build.sh --experimental --run_example abs eager cust --vendor_name=custom
```

其中：
- `${op}`：表示待执行算子，算子名小写下划线形式，如abs。  
- `${mode}`：表示执行模式，目前支持eager（aclnn调用）、graph（图模式调用）。  
- `${pkg_mode}`：表示包模式，目前仅支持cust，即自定义算子包。  
- `${vendor_name}`（可选）：与构建的自定义算子包设置一致，默认名为custom。  
- `${soc_version}`（可选）：表示NPU型号。  
- `${experimental}`（可选）：表示执行用户保存在experimental贡献目录下的算子。  

执行以下指令，进行算子的编译运行

In [ ]:
!source ${HOME}/vendors/custom_math/bin/set_env.bash;bash build.sh --experimental --run_example arange eager cust --vendor_name=custom

样例执行后会打印结果，若无报错，则证明运行成功。

---

## **课后实践**
以下是experimental/math/arange算子的整体验证过程

In [ ]:
!git clone https://gitcode.com/cann/ops-math.git    
%cd ops-math 
!bash build.sh --pkg --experimental --vendor_name=custom --soc=ascend910b --ops=arange -j16  
!./ops-math/build_out/cann-ops-math-custom_linux-aarch64.run    
!bash build.sh --experimental --run_example arange eager cust --vendor_name=custom

请修改以上内容，编译并运行math/abs算子，正常运行后结果如下
```
abs result[0] is: 1.000000
abs result[1] is: 1.000000
abs result[2] is: 1.000000
abs result[3] is: 2.000000
abs result[4] is: 2.000000
abs result[5] is: 2.000000
abs result[6] is: 3.000000
abs result[7] is: 3.000000
```

**执行以下代码获取答案**

In [ ]:
!cat ./answer/06.02_answer.txt